# CLI

Command-line interface for snapshotting node rewards and exporting reward-withdrawal transactions.

In [1]:
#| default_exp cli

## CLI definitions

Expose subcommands for snapshotting and transaction export.

In [2]:
#| export
from __future__ import annotations

import argparse

from nym_node_reward_tracker.snapshot import run_snapshot
from nym_node_reward_tracker.reward_transactions import run_reward_transactions
from nym_node_reward_tracker.cache import run_cache
from nym_node_reward_tracker.epoch_by_epoch import run_epoch_by_epoch


_DEF_SNAPSHOT_HELP = "Snapshot current rewards and update 7/30-day history"
_DEF_REWARD_TX_HELP = "Export reward-withdrawal transactions with fiat valuation"
_DEF_CACHE_HELP = "Build/update SQLite cache for epoch-by-epoch reward reconstruction"
_DEF_EPOCH_HELP = "Replay node rewards epoch-by-epoch from cache events and export Excel sheets"


def _add_snapshot_parser(subparsers: argparse._SubParsersAction) -> None:
    p = subparsers.add_parser(
        "snapshot",
        help=_DEF_SNAPSHOT_HELP,
        description=_DEF_SNAPSHOT_HELP,
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--data-dir", default="data", help="Directory for inputs/outputs")
    p.add_argument("--wallets", default="wallet-addresses.csv", help="CSV with columns: address, tag")
    p.add_argument("--out", dest="out_csv", default="node-balances.csv", help="Output file (.csv or .xlsx)")
    p.add_argument("--history", default="data.yaml", help="History YAML filename")
    p.add_argument("--source", default="spectre", choices=["spectre", "cosmos"], help="Snapshot data source")
    p.add_argument("--spectre-nodes-url", default=None, help="Override Spectre nodes API URL")
    p.add_argument("--validator-bonded-url", default=None, help="Override validator bonded API URL")
    p.add_argument("--validator-described-url", default=None, help="Override validator described API URL")
    p.add_argument("--spectre-balance-url", default=None, help="Override Spectre balance URL template")
    p.add_argument("--log-file", default=None, help="Optional log file path")
    p.add_argument("--log-level", default="INFO", help="Log level")
    p.set_defaults(_cmd="snapshot")


def _add_reward_tx_parser(subparsers: argparse._SubParsersAction) -> None:
    p = subparsers.add_parser(
        "reward-transactions",
        help=_DEF_REWARD_TX_HELP,
        description=_DEF_REWARD_TX_HELP,
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--wallets", default="wallet-addresses.csv", help="CSV with columns: address, tag")
    p.add_argument("--out", default="nym_rewards_tax_export.csv", help="Output file (.csv or .xlsx)")
    p.add_argument("--currency", default="eur", choices=["eur", "usd", "gbp", "chf"], help="Target fiat currency")
    p.add_argument("--start", default=None, help="Start date (YYYY-MM-DD), UTC inclusive")
    p.add_argument("--end", default=None, help="End date (YYYY-MM-DD), UTC inclusive")
    p.add_argument("--nyx-api", default=None, help="Nyx Cosmos API base URL")
    p.add_argument("--mixnet-contract", default=None, help="Mixnet contract address")
    p.add_argument("--coin-id", default=None, help="CoinGecko coin id")
    p.add_argument("--coingecko-base", default=None, help="CoinGecko API base URL")
    p.add_argument("--coingecko-api-key", default=None, help="CoinGecko API key")
    p.add_argument("--cache-dir", default=".cache", help="Cache directory (set empty to disable)")
    p.add_argument("--search-mode", default="sender", choices=["sender", "recipient"], help="Tx search strategy")
    p.add_argument("--explorer-base", default=None, help="Base URL for tx links (defaults to Nyx tx REST)")
    p.add_argument("--tx-api-base", default=None, help="Base URL for direct tx API lookups (default: Nyx /txs endpoint)")
    p.add_argument("--tx-rpc-base", default=None, help="Base URL for RPC tx lookups (default: https://rpc.nymtech.net)")
    p.add_argument("--log-file", default="nym_tax_rewards_export.log", help="Path to log file")
    p.add_argument(
        "--log-level",
        default="INFO",
        choices=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"],
        help="Log level for console and file logging",
    )
    p.set_defaults(_cmd="reward-transactions")


def _add_cache_parser(subparsers: argparse._SubParsersAction) -> None:
    p = subparsers.add_parser(
        "cache",
        help=_DEF_CACHE_HELP,
        description=_DEF_CACHE_HELP,
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--data-dir", default="data", help="Directory containing wallets CSV")
    p.add_argument("--wallets", default="wallet-addresses.csv", help="CSV with columns: address, tag")
    p.add_argument("--db-path", default="data/nym_cache.sqlite", help="SQLite cache path")
    p.add_argument("--window-size", type=int, default=None, help="Override height window size for all event kinds")
    p.add_argument("--window-size-reward", type=int, default=10000, help="Reward scan window size")
    p.add_argument("--window-size-delegation", type=int, default=10000, help="Delegation scan window size")
    p.add_argument("--window-size-withdraw", type=int, default=20000, help="Withdraw scan window size")
    p.add_argument("--wallet-window-size", type=int, default=100000, help="Wallet-actions scan window size")
    p.add_argument("--wallet-start-height", type=int, default=0, help="Lower bound for wallet/node scans (0 means auto)")
    p.add_argument("--page-size", type=int, default=100, help="Tx search page size")
    p.add_argument("--max-pages", type=int, default=200, help="Max pages per query")
    p.add_argument("--timeout", type=int, default=20, help="HTTP timeout in seconds")
    p.add_argument("--force", action="store_true", help="Force rescan even if window is marked ok")
    p.add_argument("--mixnet-contract", default=None, help="Override mixnet contract address")
    p.add_argument("--log-file", default="nym_cache.log", help="Path to log file")
    p.add_argument("--log-level", default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"], help="Log level")
    p.set_defaults(_cmd="cache")


def _add_epoch_parser(subparsers: argparse._SubParsersAction) -> None:
    p = subparsers.add_parser(
        "epoch-by-epoch",
        help=_DEF_EPOCH_HELP,
        description=_DEF_EPOCH_HELP,
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--data-dir", default="data", help="Directory containing wallets CSV and default output")
    p.add_argument("--wallets", default="wallet-addresses.csv", help="CSV with columns: address, tag")
    p.add_argument("--out", default="epoch_by_epoch_rewards.xlsx", help="Excel output path (.xlsx)")
    p.add_argument("--db-path", default="data/nym_cache.sqlite", help="SQLite cache path")
    p.add_argument("--mixnet-contract", default=None, help="Override mixnet contract address")
    p.add_argument("--window-size", type=int, default=None, help="Override height window size for all event kinds")
    p.add_argument("--window-size-reward", type=int, default=10000, help="Reward scan window size")
    p.add_argument("--window-size-delegation", type=int, default=10000, help="Delegation scan window size")
    p.add_argument("--window-size-withdraw", type=int, default=20000, help="Withdraw scan window size")
    p.add_argument("--wallet-window-size", type=int, default=100000, help="Wallet-actions scan window size")
    p.add_argument("--wallet-start-height", type=int, default=0, help="Lower bound for wallet/node scans (0 means auto)")
    p.add_argument("--page-size", type=int, default=100, help="Tx search page size")
    p.add_argument("--max-pages", type=int, default=200, help="Max pages per query")
    p.add_argument("--timeout", type=int, default=20, help="HTTP timeout in seconds")
    p.add_argument("--force", action="store_true", help="Force rescan even if window is marked ok")
    p.add_argument("--log-file", default="nym_epoch_by_epoch.log", help="Path to log file")
    p.add_argument("--log-level", default="INFO", choices=["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"], help="Log level")
    p.set_defaults(_cmd="epoch-by-epoch")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Track Nym node rewards and export reward transactions.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    subparsers = parser.add_subparsers(dest="command", required=True)
    _add_snapshot_parser(subparsers)
    _add_reward_tx_parser(subparsers)
    _add_cache_parser(subparsers)
    _add_epoch_parser(subparsers)
    return parser


def _run_snapshot_from_args(args: argparse.Namespace) -> int:
    run_snapshot(
        data_dir=args.data_dir,
        wallets_csv=args.wallets,
        out_csv=args.out_csv,
        hist_file=args.history,
        source=args.source,
        spectre_nodes_url=args.spectre_nodes_url,
        validator_bonded_url=args.validator_bonded_url,
        validator_described_url=args.validator_described_url,
        spectre_balance_url=args.spectre_balance_url,
        log_file=args.log_file,
        log_level=args.log_level,
    )
    return 0


def _run_reward_tx_from_args(args: argparse.Namespace) -> int:
    return run_reward_transactions(
        wallets_csv=args.wallets,
        out_csv=args.out,
        currency=args.currency,
        start=args.start,
        end=args.end,
        nyx_api_base=args.nyx_api,
        mixnet_contract=args.mixnet_contract,
        coin_id=args.coin_id,
        coingecko_base=args.coingecko_base,
        coingecko_api_key=args.coingecko_api_key,
        cache_dir=args.cache_dir,
        search_mode=args.search_mode,
        explorer_base=args.explorer_base,
        tx_api_base=args.tx_api_base,
        tx_rpc_base=args.tx_rpc_base,
        log_file=args.log_file,
        log_level=args.log_level,
    )


def _run_cache_from_args(args: argparse.Namespace) -> int:
    return run_cache(
        data_dir=args.data_dir,
        wallets_csv=args.wallets,
        db_path=args.db_path,
        window_size=args.window_size,
        window_size_reward=args.window_size_reward,
        window_size_delegation=args.window_size_delegation,
        window_size_withdraw=args.window_size_withdraw,
        wallet_window_size=args.wallet_window_size,
        force=args.force,
        wallet_start_height=args.wallet_start_height,
        page_size=args.page_size,
        max_pages=args.max_pages,
        timeout=args.timeout,
        contract=args.mixnet_contract or "n17srjznxl9dvzdkpwpw24gg668wc73val88a6m5ajg6ankwvz9wtst0cznr",
        log_file=args.log_file,
        log_level=args.log_level,
    )


def _run_epoch_by_epoch_from_args(args: argparse.Namespace) -> int:
    return run_epoch_by_epoch(
        data_dir=args.data_dir,
        wallets_csv=args.wallets,
        out_xlsx=args.out,
        db_path=args.db_path,
        window_size=args.window_size,
        window_size_reward=args.window_size_reward,
        window_size_delegation=args.window_size_delegation,
        window_size_withdraw=args.window_size_withdraw,
        wallet_window_size=args.wallet_window_size,
        wallet_start_height=args.wallet_start_height,
        page_size=args.page_size,
        max_pages=args.max_pages,
        timeout=args.timeout,
        force=args.force,
        contract=args.mixnet_contract or "n17srjznxl9dvzdkpwpw24gg668wc73val88a6m5ajg6ankwvz9wtst0cznr",
        log_file=args.log_file,
        log_level=args.log_level,
    )


def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.command == "snapshot":
        return _run_snapshot_from_args(args)
    if args.command == "reward-transactions":
        return _run_reward_tx_from_args(args)
    if args.command == "cache":
        return _run_cache_from_args(args)
    if args.command == "epoch-by-epoch":
        return _run_epoch_by_epoch_from_args(args)
    parser.print_help()
    return 2


In [3]:
# verify_cli_parser
p = build_parser()
a = p.parse_args(["cache"])
assert a.command == "cache"
assert a.window_size_reward == 10000


In [4]:
#| hide
import nbdev; nbdev.nbdev_export()
